# GNSS Data Parser and Time Synchronization

## 1. Setup and Configuration

This initial cell imports the required libraries (`os`, `pandas`, `numpy`) and defines the file paths for our input data and output artifacts. This ensures that our notebook is organized and that all file I/O operations are handled consistently.

In [1]:
import os
import pandas as pd
import numpy as np

# --- File Architecture ---
# Define relative paths for robust file I/O.
BASE_DIR = "../../GNSS_Multipath_Project" 
INPUT_DIR = os.path.join(BASE_DIR, "data/01_raw/2020-09-04-US-SF-1/")
OUTPUT_DIR = os.path.join(BASE_DIR, "data/02_interim/")

# Ensure the output directory exists.
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- File Paths ---
gnss_log_path = os.path.join(INPUT_DIR, "Pixel4_GnssLog.txt")
chipset_nmea_path = os.path.join(INPUT_DIR, "Pixel4_chipset.nmea")
span_nmea_path = os.path.join(INPUT_DIR, "SPAN_Pixel4_10Hz.nmea")
output_csv_path = os.path.join(OUTPUT_DIR, "time_synced_epochs.csv")

print(f"Input GNSS Log: {gnss_log_path}")
print(f"Input Chipset NMEA: {chipset_nmea_path}")
print(f"Input SPAN NMEA: {span_nmea_path}")
print(f"Output CSV: {output_csv_path}")

Input GNSS Log: ../../GNSS_Multipath_Project\data/01_raw/2020-09-04-US-SF-1/Pixel4_GnssLog.txt
Input Chipset NMEA: ../../GNSS_Multipath_Project\data/01_raw/2020-09-04-US-SF-1/Pixel4_chipset.nmea
Input SPAN NMEA: ../../GNSS_Multipath_Project\data/01_raw/2020-09-04-US-SF-1/SPAN_Pixel4_10Hz.nmea
Output CSV: ../../GNSS_Multipath_Project\data/02_interim/time_synced_epochs.csv


## Step 1: Parse Raw Android GNSS Log

This function, `parse_gnss_log`, is designed to read the raw Android GNSS measurements from `Pixel4_GnssLog.txt`. It specifically extracts rows marked as "Raw", which contain the detailed satellite signal data. We then select a subset of columns relevant for multipath analysis and load them into a Pandas DataFrame. This initial parsing is the first step in transforming the raw text data into a structured format suitable for analysis.

In [2]:
def parse_gnss_log(file_path: str) -> pd.DataFrame:
    """
    Parses the raw Android GNSS log file and extracts relevant columns.

    Args:
        file_path: The full path to the GnssLog.txt file.

    Returns:
        A Pandas DataFrame containing the parsed "Raw" GNSS measurements.
    """
    with open(file_path, 'r') as f:
        lines = f.readlines()

    # Extract only the "Raw" measurements, ignoring the header.
    raw_lines = [line.strip().split(',') for line in lines if line.startswith('Raw')]
    
    # The first line after the header contains the column names for "Raw"
    header_line = [line.strip() for line in lines if line.startswith('# Raw')][0]
    columns = header_line.replace('# Raw,', '').split(',')
    # The "Raw" identifier is a column itself, so we add it to the list of columns.
    columns.insert(0, 'Raw')
    
    df = pd.DataFrame(raw_lines, columns=columns)

    # Keep only the specified columns for our analysis.
    required_columns = [
        'TimeNanos', 'FullBiasNanos', 'State', 'ReceivedSvTimeUncertaintyNanos',
        'Cn0DbHz', 'MultipathIndicator', 'AccumulatedDeltaRangeState'
    ]
    df = df[required_columns]

    # Convert columns to appropriate numeric types
    for col in df.columns:
        # Use Int64 (nullable) for time columns to handle large numbers and potential NaNs
        if col in ['TimeNanos', 'FullBiasNanos']:
             df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
        else:
             df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

# --- Execute Parsing ---
gnss_df = parse_gnss_log(gnss_log_path)
print("Successfully parsed GNSS log.")
print(f"DataFrame shape: {gnss_df.shape}")
gnss_df.head()

Successfully parsed GNSS log.
DataFrame shape: (47003, 7)


,TimeNanos,FullBiasNanos,State,ReceivedSvTimeUncertaintyNanos,Cn0DbHz,MultipathIndicator,AccumulatedDeltaRangeState
0,1825976000000,-1283272826466003753,16399,5,40.5,0,25
1,1825976000000,-1283272826466003753,16399,7,37.8,0,25
2,1825976000000,-1283272826466003753,16399,6,38.3,0,25
3,1825976000000,-1283272826466003753,16399,27,24.1,0,21
4,1825976000000,-1283272826466003753,16399,35,22.9,0,19


## Step 2: Implement Hard-Coded Outlier Rejection

This cell applies Google's official data-cleaning standards to the GNSS data. We filter out rows that are likely to be erroneous or unreliable based on several criteria, such as invalid time measurements and specific satellite state flags. This is a critical preprocessing step to ensure the quality of our data before we proceed with time synchronization and feature engineering. Each condition is checked, and rows that meet any of the rejection criteria are dropped from the DataFrame.

In [3]:
def apply_outlier_rejection(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies hard-coded outlier rejection criteria to the GNSS DataFrame.

    Args:
        df: The input DataFrame with raw GNSS measurements.

    Returns:
        A cleaned DataFrame with outliers removed.
    """
    initial_rows = len(df)
    
    # 1. Reject if FullBiasNanos is zero, NaN, or invalid.
    df.dropna(subset=['FullBiasNanos'], inplace=True)
    df = df[df['FullBiasNanos'] != 0]
    
    # 2. Reject if TimeNanos is empty or negative.
    df.dropna(subset=['TimeNanos'], inplace=True)
    df = df[df['TimeNanos'] > 0]

    # 3. Reject based on State flags (must have TOW_DECODED or TOW_KNOWN).
    # STATE_TOW_DECODED is bit 3 (value 8)
    # STATE_TOW_KNOWN is bit 14 (value 16384)
    state_tow_decoded = (df['State'] & (1 << 3)) != 0
    state_tow_known = (df['State'] & (1 << 14)) != 0
    df = df[state_tow_decoded | state_tow_known]

    # 4. Reject if ReceivedSvTimeUncertaintyNanos > 500.
    df = df[df['ReceivedSvTimeUncertaintyNanos'] <= 500]
    
    final_rows = len(df)
    print(f"Rejected {initial_rows - final_rows} rows out of {initial_rows}.")
    print(f"Remaining rows: {final_rows}")
    
    return df

# --- Execute Cleaning ---
cleaned_gnss_df = apply_outlier_rejection(gnss_df.copy())
cleaned_gnss_df.head()

Rejected 54 rows out of 47003.
Remaining rows: 46949


,TimeNanos,FullBiasNanos,State,ReceivedSvTimeUncertaintyNanos,Cn0DbHz,MultipathIndicator,AccumulatedDeltaRangeState
0,1825976000000,-1283272826466003753,16399,5,40.5,0,25
1,1825976000000,-1283272826466003753,16399,7,37.8,0,25
2,1825976000000,-1283272826466003753,16399,6,38.3,0,25
3,1825976000000,-1283272826466003753,16399,27,24.1,0,21
4,1825976000000,-1283272826466003753,16399,35,22.9,0,19


## Step 3: Parse NMEA Files and Synchronize Time

This section handles the parsing of NMEA files from both the smartphone's chipset and the high-accuracy SPAN system (our ground truth). We then create a common time feature (`GpsTime`) in our main GNSS log. Finally, we use `pandas.merge_asof` to perform a nearest-neighbor time synchronization, merging the chipset and SPAN location data with the satellite signal data. This aligns all our data sources to a common timeline, which is essential for calculating position errors accurately.

In [4]:
def nmea_to_decimal_degrees(coord: str, direction: str) -> float:
    """Converts NMEA coordinate format (DDMM.MMMM) to decimal degrees."""
    if not coord or not direction:
        return np.nan
    
    coord = float(coord)
    degrees = int(coord / 100)
    minutes = coord - degrees * 100
    decimal_degrees = degrees + minutes / 60
    
    if direction in ['S', 'W']:
        decimal_degrees *= -1
        
    return decimal_degrees

def parse_nmea_file(file_path: str) -> pd.DataFrame:
    """
    Parses NMEA files to extract UTC time, latitude, and longitude from GGA sentences.
    """
    gga_lines = []
    with open(file_path, 'r') as f:
        for line in f:
            if '$GPGGA' in line or '$GNGGA' in line:
                parts = line.strip().split(',')
                # Ensure the line has enough parts for lat/lon/time
                if len(parts) > 5 and parts[1] and parts[2] and parts[4]:
                    gga_lines.append(parts)

    if not gga_lines:
        return pd.DataFrame()

    df = pd.DataFrame(gga_lines)
    # Keep only relevant columns for time, lat, lon
    df = df[[1, 2, 3, 4, 5]]
    df.columns = ['utc_time', 'lat', 'lat_dir', 'lon', 'lon_dir']

    # Convert to numeric and handle potential errors
    df['utc_time'] = pd.to_numeric(df['utc_time'], errors='coerce')
    df['lat_decimal'] = df.apply(lambda row: nmea_to_decimal_degrees(row['lat'], row['lat_dir']), axis=1)
    df['lon_decimal'] = df.apply(lambda row: nmea_to_decimal_degrees(row['lon'], row['lon_dir']), axis=1)

    # Drop rows where conversion failed
    df.dropna(subset=['utc_time', 'lat_decimal', 'lon_decimal'], inplace=True)
    
    return df[['utc_time', 'lat_decimal', 'lon_decimal']]

# --- Parse NMEA Files ---
chipset_df = parse_nmea_file(chipset_nmea_path)
chipset_df.rename(columns={'lat_decimal': 'lat_chipset', 'lon_decimal': 'lon_chipset'}, inplace=True)

span_df = parse_nmea_file(span_nmea_path)
span_df.rename(columns={'lat_decimal': 'lat_span', 'lon_decimal': 'lon_span'}, inplace=True)

print("Parsed Chipset NMEA data:")
print(chipset_df.head())
print("\nParsed SPAN NMEA data:")
print(span_df.head())

# --- Time Synchronization ---
LEAP_SECONDS = 18
NANOS_IN_SECOND = 1e9
SECONDS_IN_WEEK = 604800

# 1. Calculate GpsTimeNanos for the raw GNSS log.
# This is the number of nanoseconds since the beginning of the GPS week.
# We use modulo to get the time of week, which is a robust way to handle the time reference.
raw_gps_time_ns = cleaned_gnss_df['TimeNanos'] - cleaned_gnss_df['FullBiasNanos']
cleaned_gnss_df['GpsTimeNanos'] = raw_gps_time_ns % (SECONDS_IN_WEEK * NANOS_IN_SECOND)

# 2. Convert NMEA UTC time to a comparable GPS time of week.
def utc_time_to_gps_time_of_week(utc_time: pd.Series, day_of_week: int) -> pd.Series:
    """
    Converts UTC time from NMEA to GPS time of the week in nanoseconds.
    
    Args:
        utc_time: A Pandas Series containing UTC time in HHMMSS.ss format.
        day_of_week: The day of the week (0=Sunday, 1=Monday, ..., 6=Saturday).
        
    Returns:
        A Pandas Series containing GPS time of the week in nanoseconds.
    """
    t_float = utc_time.astype(float)
    hours = (t_float // 10000).astype(np.int64)
    minutes = ((t_float // 100) % 100).astype(np.int64)
    seconds = t_float % 100
    
    # Nanoseconds into the day
    nanos_of_day = (hours * 3600 + minutes * 60 + seconds) * NANOS_IN_SECOND
    
    # Nanoseconds into the week
    # Day of week for 2020-05-14 is Thursday (4)
    nanos_of_week = (day_of_week * 24 * 3600 * NANOS_IN_SECOND) + nanos_of_day
    
    # Convert to GPS time of week
    gps_time_of_week = (nanos_of_week + (LEAP_SECONDS * NANOS_IN_SECOND)) % (SECONDS_IN_WEEK * NANOS_IN_SECOND)
    
    return gps_time_of_week

# The date is 2020-09-04, which is a Friday. In Python, Friday is day 4 (if Monday is 0)
# or 5 if Sunday is 0. Let's assume Sunday=0, so Friday=5.
DAY_OF_WEEK = 5 
chipset_df['GpsTimeNanos'] = utc_time_to_gps_time_of_week(chipset_df['utc_time'], DAY_OF_WEEK)
span_df['GpsTimeNanos'] = utc_time_to_gps_time_of_week(span_df['utc_time'], DAY_OF_WEEK)


# 3. Ensure data types are consistent before merging.
for df in [cleaned_gnss_df, chipset_df, span_df]:
    df['GpsTimeNanos'] = df['GpsTimeNanos'].astype(np.int64)
    df.sort_values('GpsTimeNanos', inplace=True)

# 4. Perform the merge with a tolerance.
TOLERANCE_NS = int(0.5 * 1e9)  # 0.5 seconds

merged_df = pd.merge_asof(
    left=cleaned_gnss_df,
    right=chipset_df.drop(columns=['utc_time']),
    on='GpsTimeNanos',
    direction='nearest',
    tolerance=TOLERANCE_NS
)

merged_df = pd.merge_asof(
    left=merged_df,
    right=span_df.drop(columns=['utc_time']),
    on='GpsTimeNanos',
    direction='nearest',
    tolerance=TOLERANCE_NS
)

print("\nShape of merged DataFrame:", merged_df.shape)
merged_df.head()

Parsed Chipset NMEA data:
   utc_time  lat_chipset  lon_chipset
0  171035.0    37.416610  -122.082001
1  171036.0    37.416610  -122.082001
2  171037.0    37.416610  -122.082001
3  171038.0    37.416610  -122.082001
4  171039.0    37.416609  -122.082002

Parsed SPAN NMEA data:
   utc_time   lat_span    lon_span
0  170513.0  37.416622 -122.082051
1  170513.1  37.416622 -122.082051
2  170513.2  37.416622 -122.082051
3  170513.3  37.416622 -122.082051
4  170513.4  37.416622 -122.082051

Shape of merged DataFrame: (46949, 12)


,TimeNanos,FullBiasNanos,State,ReceivedSvTimeUncertaintyNanos,Cn0DbHz,MultipathIndicator,AccumulatedDeltaRangeState,GpsTimeNanos,lat_chipset,lon_chipset,lat_span,lon_span
0,1825976000000,-1283272826466003753,16399,5,40.5,0,25,493852442003712,NaN,NaN,37.41662,-122.082045
1,1825976000000,-1283272826466003753,16399,14,30.2,0,25,493852442003712,NaN,NaN,37.41662,-122.082045
2,1825976000000,-1283272826466003753,23567,17,28.8,0,25,493852442003712,NaN,NaN,37.41662,-122.082045
3,1825976000000,-1283272826466003753,23567,6,38.9,0,25,493852442003712,NaN,NaN,37.41662,-122.082045
4,1825976000000,-1283272826466003753,23567,27,24.0,0,16,493852442003712,NaN,NaN,37.41662,-122.082045


## Step 4: Calculate Position Error (Haversine Distance)

Now that our data is time-synchronized, we can calculate the positioning error. This is done by computing the Haversine distance between the ground truth coordinates (from the SPAN system) and the estimated coordinates (from the phone's chipset). The Haversine formula is used to determine the great-circle distance between two points on a sphere, which gives us a highly accurate measure of the error in meters. This error will be the basis for creating our target label.

In [5]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance between two points 
    on the earth (specified in decimal degrees).
    
    Vectorized implementation for Pandas.
    """
    # Earth radius in meters
    R = 6371000
    
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    distance = R * c
    
    return distance

# --- Clean up rows with no coordinate data ---
merged_df.dropna(subset=['lat_span', 'lon_span', 'lat_chipset', 'lon_chipset'], inplace=True)


# --- Calculate Position Error ---
merged_df['Position_Error_Meters'] = haversine_distance(
    merged_df['lat_span'],
    merged_df['lon_span'],
    merged_df['lat_chipset'],
    merged_df['lon_chipset']
)

print("Calculated Position Error (Haversine Distance).")
print(merged_df[['lat_span', 'lon_span', 'lat_chipset', 'lon_chipset', 'Position_Error_Meters']].head())
merged_df['Position_Error_Meters'].describe()

Calculated Position Error (Haversine Distance).
    lat_span    lon_span  lat_chipset  lon_chipset  Position_Error_Meters
40  37.41662 -122.082045     37.41661  -122.082001               4.066774
41  37.41662 -122.082045     37.41661  -122.082001               4.066774
42  37.41662 -122.082045     37.41661  -122.082001               4.066774
43  37.41662 -122.082045     37.41661  -122.082001               4.066774
44  37.41662 -122.082045     37.41661  -122.082001               4.066774


count    46909.000000
mean         8.424349
std          2.709380
min          0.073211
25%          6.720052
50%          8.819303
75%         10.304842
max         16.199109
Name: Position_Error_Meters, dtype: float64

In [6]:
# --- Diagnostic Check ---
# Check how many unique coordinates actually exist in the merged dataframe.
# If the number is very low (e.g., 1), it indicates the merge process may have failed
# to align the time series correctly, causing one set of coordinates to be
# broadcast across all rows.

print("--- Coordinate Uniqueness Diagnostic ---")
print(f"Unique SPAN Latitudes: {merged_df['lat_span'].nunique()}")
print(f"Unique SPAN Longitudes: {merged_df['lon_span'].nunique()}")
print(f"Unique Chipset Latitudes: {merged_df['lat_chipset'].nunique()}")
print(f"Unique Chipset Longitudes: {merged_df['lon_chipset'].nunique()}")
print(f"Unique Position Error Meters: {merged_df['Position_Error_Meters'].nunique()}")

print("\n--- Raw Coordinate Preview (first 5 rows) ---")
print(merged_df[['lat_span', 'lon_span', 'lat_chipset', 'lon_chipset']].head())



--- Coordinate Uniqueness Diagnostic ---
Unique SPAN Latitudes: 1648
Unique SPAN Longitudes: 1652
Unique Chipset Latitudes: 1644
Unique Chipset Longitudes: 1640
Unique Position Error Meters: 1679

--- Raw Coordinate Preview (first 5 rows) ---
    lat_span    lon_span  lat_chipset  lon_chipset
40  37.41662 -122.082045     37.41661  -122.082001
41  37.41662 -122.082045     37.41661  -122.082001
42  37.41662 -122.082045     37.41661  -122.082001
43  37.41662 -122.082045     37.41661  -122.082001
44  37.41662 -122.082045     37.41661  -122.082001


## Step 5: Create Target Label and Export

In this final step, we create our binary classification target, `is_multipath`. Based on the calculated `Position_Error_Meters`, we label each epoch as either multipath-affected (1) or not (0) using a 10-meter threshold. We then drop any rows with missing values that may have resulted from the time-synchronization merge. Finally, the cleaned, synchronized, and labeled DataFrame is exported to a CSV file in the `02_interim` directory, ready for the next phase of the analysis pipeline: exploratory data analysis and model training.

In [7]:
# --- Create Target Label ---
# We define our binary target 'is_multipath' based on a position error threshold.
# If the error is greater than 10 meters, we classify it as a multipath event.
MULTIPATH_THRESHOLD_METERS = 10.0
merged_df['is_multipath'] = (merged_df['Position_Error_Meters'] > MULTIPATH_THRESHOLD_METERS).astype(int)

# --- Final Cleanup and Export ---
# Drop rows with NaN values that may have been introduced during the merge.
final_df = merged_df.dropna().copy()

# Display the value counts for our new target variable.
print("Target Label Distribution ('is_multipath'):")
print(final_df['is_multipath'].value_counts())

# Export the final DataFrame to the interim data folder.
final_df.to_csv(output_csv_path, index=False)

print(f"\nSuccessfully created and saved the final DataFrame.")
print(f"Final DataFrame shape: {final_df.shape}")
print(f"Data saved to: {output_csv_path}")

final_df.head()

Target Label Distribution ('is_multipath'):
is_multipath
0    33512
1    13397
Name: count, dtype: int64

Successfully created and saved the final DataFrame.
Final DataFrame shape: (46909, 14)
Data saved to: ../../GNSS_Multipath_Project\data/02_interim/time_synced_epochs.csv

Successfully created and saved the final DataFrame.
Final DataFrame shape: (46909, 14)
Data saved to: ../../GNSS_Multipath_Project\data/02_interim/time_synced_epochs.csv


,TimeNanos,FullBiasNanos,State,ReceivedSvTimeUncertaintyNanos,Cn0DbHz,MultipathIndicator,AccumulatedDeltaRangeState,GpsTimeNanos,lat_chipset,lon_chipset,lat_span,lon_span,Position_Error_Meters,is_multipath
40,1826976000000,-1283272826466003748,16399,61,18.2,0,16,493853442003712,37.41661,-122.082001,37.41662,-122.082045,4.066774,0
41,1826976000000,-1283272826466003748,23567,31,23.9,0,16,493853442003712,37.41661,-122.082001,37.41662,-122.082045,4.066774,0
42,1826976000000,-1283272826466003748,23567,27,24.4,0,16,493853442003712,37.41661,-122.082001,37.41662,-122.082045,4.066774,0
43,1826976000000,-1283272826466003748,16399,4,43.1,0,25,493853442003712,37.41661,-122.082001,37.41662,-122.082045,4.066774,0
44,1826976000000,-1283272826466003748,23567,17,28.5,0,25,493853442003712,37.41661,-122.082001,37.41662,-122.082045,4.066774,0


In [8]:
import folium
from folium.plugins import AntPath

# --- Create an Interactive Map to Visualize the Route ---

# Ensure the final_df has coordinate data
if not final_df.empty and 'lat_span' in final_df.columns:
    # Get the center point for the map
    map_center = [final_df['lat_span'].mean(), final_df['lon_span'].mean()]

    # Create a base map
    # 'cartodbpositron' is a clean, light-colored map. 'OpenStreetMap' is another good default.
    m = folium.Map(location=map_center, zoom_start=16, tiles='cartodbpositron')

    # Add a satellite view layer option
    folium.TileLayer(
        'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri',
        name='Esri Satellite',
        overlay=False,
        control=True
    ).add_to(m)

    # --- Prepare Coordinate Data for Plotting ---
    # Create lists of (latitude, longitude) tuples
    ground_truth_points = list(zip(final_df['lat_span'], final_df['lon_span']))
    chipset_points = list(zip(final_df['lat_chipset'], final_df['lon_chipset']))

    # --- Add Lines to the Map ---
    # Add the ground truth path (SPAN) in blue
    folium.PolyLine(
        ground_truth_points,
        color='blue',
        weight=5,
        opacity=0.8,
        popup='Ground Truth (SPAN)'
    ).add_to(m)

    # Add the phone's estimated path (Chipset) in red
    folium.PolyLine(
        chipset_points,
        color='red',
        weight=5,
        opacity=0.8,
        popup='Phone Estimate (Chipset)'
    ).add_to(m)

    # --- Add Start and End Markers ---
    folium.Marker(
        location=ground_truth_points[0],
        popup='Start',
        icon=folium.Icon(color='green', icon='play')
    ).add_to(m)

    folium.Marker(
        location=ground_truth_points[-1],
        popup='End',
        icon=folium.Icon(color='red', icon='stop')
    ).add_to(m)
    
    # Add a layer control to switch between map styles
    folium.LayerControl().add_to(m)

    # Display the map
    display(m)
else:
    print("Final DataFrame is empty or does not contain coordinate columns. Cannot generate map.")